# US Species Selection Pipeline

This notebook generalizes the California-focused EDA in `01_eda.ipynb` to the full United States.

**Goal:** Identify plant species that are strong candidates for a future multimodal image + geo + season classification dataset.

**What this notebook does:**
1. Loads a US-wide GBIF metadata export and filters to target families
2. Computes per-species summary statistics (count, temporal coverage, geographic spread)
3. Compares three selection strategies: `top_n`, `time_diverse`, `time_geo_diverse`
4. Produces candidate species lists and exports summary CSVs

## 0. Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=FutureWarning)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


## 1. Configuration

Edit parameters here. Everything downstream reads from these variables.

In [ ]:
# ─────────────────────────────────────────────────────────
# Data
# ─────────────────────────────────────────────────────────
DATA_PATH  = '../data/US_plants_gbif.csv'   # GBIF export CSV (tab- or comma-separated)
OUTPUT_DIR = '../data/selections_us/'        # where CSVs will be saved

# ─────────────────────────────────────────────────────────
# Geographic scope
# ─────────────────────────────────────────────────────────
COUNTRY = 'US'   # filter on countryCode; set None to skip

# ─────────────────────────────────────────────────────────
# Target families  (swap freely — nothing below is hardcoded)
# ─────────────────────────────────────────────────────────
TARGET_FAMILIES = ['Asteraceae', 'Poaceae', 'Apiaceae']

# Optional extra families to load alongside targets (excluded from
# selection logic but available for comparison if desired).
OPTIONAL_EXTRA_FAMILIES = []   # e.g. ['Fabaceae', 'Lamiaceae']

# ─────────────────────────────────────────────────────────
# Observation thresholds
# ─────────────────────────────────────────────────────────
MIN_OBS_PER_SPECIES       = 300   # minimum raw observations to be eligible
MIN_MEDIA_OBS_PER_SPECIES =  50   # minimum image-backed obs (mediaType == StillImage)
MIN_MONTHS                =   3   # species must appear in at least this many distinct months

# ─────────────────────────────────────────────────────────
# Selection
# ─────────────────────────────────────────────────────────
TOP_K_PER_FAMILY = 15    # how many species to pick per family per strategy
RANDOM_STATE     = 42

# ─────────────────────────────────────────────────────────
# Optional flags
# ─────────────────────────────────────────────────────────
RESEARCH_GRADE_ONLY = False   # True → keep only basisOfRecord == HUMAN_OBSERVATION
REQUIRE_IMAGE       = False   # True → also require mediaType == StillImage

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded. Output dir:', OUTPUT_DIR)


## 2. Helper Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# load_and_filter_gbif
#   Reads a GBIF CSV export, keeps target families, optionally filters to
#   a country code and/or research-grade / image-backed records.
# ─────────────────────────────────────────────────────────────────────────────
def load_and_filter_gbif(
    path, target_families, extra_families=None,
    country=None, research_grade_only=False, require_image=False,
):
    # GBIF exports use tab separation by default; fall back to comma
    try:
        df = pd.read_csv(path, sep='\t', low_memory=False)
        if df.shape[1] < 5:
            raise ValueError('probably not tab-separated')
    except Exception:
        df = pd.read_csv(path, low_memory=False)

    print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

    if country and 'countryCode' in df.columns:
        df = df[df['countryCode'] == country]
        print(f'After country={country}: {len(df):,} rows')

    all_families = list(target_families) + (extra_families or [])
    df = df[df['family'].isin(all_families)]
    print(f'After family filter: {len(df):,} rows')

    if research_grade_only and 'basisOfRecord' in df.columns:
        df = df[df['basisOfRecord'] == 'HUMAN_OBSERVATION']
        print(f'After research-grade filter: {len(df):,} rows')

    if require_image and 'mediaType' in df.columns:
        df = df[df['mediaType'] == 'StillImage']
        print(f'After image filter: {len(df):,} rows')

    df = df.dropna(subset=['species', 'decimalLatitude', 'decimalLongitude'])
    df = df[df['species'].str.strip() != '']
    print(f'After dropping missing species/coords: {len(df):,} rows')
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# add_time_features
#   Converts month/day/year to numeric, adds day-of-year.
#   Falls back to parsing eventDate if month/year are missing.
# ─────────────────────────────────────────────────────────────────────────────
def add_time_features(df):
    df = df.copy()
    for col in ['month', 'day', 'year']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'eventDate' in df.columns:
        parsed = pd.to_datetime(df['eventDate'], errors='coerce')
        if 'month' not in df.columns or df['month'].isna().mean() > 0.5:
            df['month'] = parsed.dt.month
        if 'year' not in df.columns or df['year'].isna().mean() > 0.5:
            df['year'] = parsed.dt.year
        if 'day' not in df.columns or df['day'].isna().mean() > 0.5:
            df['day'] = parsed.dt.day

    # DOY approximation — matches 01_eda approach
    df['doy'] = df['month'] * 30 + df['day']
    return df


# ─────────────────────────────────────────────────────────────────────────────
# summarize_species
#   Returns one row per (family, species) with count-based and
#   diversity statistics.
# ─────────────────────────────────────────────────────────────────────────────
def summarize_species(df):
    rows = []
    for (family, species), grp in df.groupby(['family', 'species']):
        count    = len(grp)
        n_months = grp['month'].dropna().nunique()
        mean_doy = grp['doy'].mean()
        std_doy  = grp['doy'].std(ddof=0) if count > 1 else 0.0

        # temporal entropy: how evenly spread across 12 months
        month_counts = grp['month'].dropna().value_counts(normalize=True)
        t_entropy = float(scipy_entropy(month_counts.values)) if len(month_counts) > 1 else 0.0

        lat_std   = grp['decimalLatitude'].std(ddof=0)  if count > 1 else 0.0
        lon_std   = grp['decimalLongitude'].std(ddof=0) if count > 1 else 0.0
        lat_range = grp['decimalLatitude'].max()  - grp['decimalLatitude'].min()
        lon_range = grp['decimalLongitude'].max() - grp['decimalLongitude'].min()
        mean_lat  = grp['decimalLatitude'].mean()
        mean_lon  = grp['decimalLongitude'].mean()
        n_states  = grp['stateProvince'].dropna().nunique() if 'stateProvince' in grp.columns else 0

        if 'mediaType' in grp.columns:
            image_count = int((grp['mediaType'] == 'StillImage').sum())
        else:
            image_count = 0

        rows.append(dict(
            family=family, species=species,
            count=count, image_count=image_count,
            n_months=n_months, t_entropy=round(t_entropy, 4),
            std_doy=round(std_doy, 2), mean_doy=round(mean_doy, 2),
            mean_lat=round(mean_lat, 4), mean_lon=round(mean_lon, 4),
            lat_std=round(lat_std, 4), lon_std=round(lon_std, 4),
            lat_range=round(lat_range, 4), lon_range=round(lon_range, 4),
            n_states=n_states,
        ))
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# filter_eligible_species
#   Apply minimum thresholds to species_summary to get the candidate pool.
# ─────────────────────────────────────────────────────────────────────────────
def filter_eligible_species(
    species_summary,
    min_obs=MIN_OBS_PER_SPECIES,
    min_media_obs=MIN_MEDIA_OBS_PER_SPECIES,
    min_months=MIN_MONTHS,
):
    mask = (
        (species_summary['count']    >= min_obs) &
        (species_summary['n_months'] >= min_months)
    )
    # only apply image threshold if the column has non-zero values
    # (it's zero when mediaType was absent from the export)
    if species_summary['image_count'].sum() > 0:
        mask &= (species_summary['image_count'] >= min_media_obs)

    eligible = species_summary[mask].copy().reset_index(drop=True)
    print(f'Eligible species: {len(eligible)} (from {len(species_summary)} total)')
    return eligible


# ─────────────────────────────────────────────────────────────────────────────
# score_species
#   Adds normalized score columns so families are comparable.
#   Scores are in [0, 1]; higher = more of that dimension.
#   Composite formula is easy to tweak — adjust weights as needed.
# ─────────────────────────────────────────────────────────────────────────────
def score_species(eligible_df):
    df = eligible_df.copy()

    def _norm(series):
        rng = series.max() - series.min()
        if rng == 0:
            return pd.Series(np.ones(len(series)), index=series.index)
        return (series - series.min()) / rng

    # normalize within-family so large families don't dominate
    for col, out in [
        ('count',    'score_count'),
        ('t_entropy', 'score_time'),
        ('n_months', 'score_months'),
    ]:
        df[out] = df.groupby('family')[col].transform(_norm)

    df['score_geo_lat'] = df.groupby('family')['lat_range'].transform(_norm)
    df['score_geo_lon'] = df.groupby('family')['lon_range'].transform(_norm)
    df['score_geo']     = (df['score_geo_lat'] + df['score_geo_lon']) / 2

    # Composite: count 40%, temporal entropy 35%, geo spread 25%
    # Adjust weights here if you want to emphasize a different dimension.
    df['score_time_geo'] = (
        df['score_count'] * 0.4 +
        df['score_time']  * 0.35 +
        df['score_geo']   * 0.25
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# select_species_by_strategy
#   Returns: selections[strategy][family] = [species, ...]
#   Three strategies mirror 01_eda:
#     top_n           — rank by raw count
#     time_diverse    — spread evenly across DOY, prefer high temporal entropy
#     time_geo_diverse— KMeans cluster on (doy, lat, lon), pick best per cluster
# ─────────────────────────────────────────────────────────────────────────────
def select_species_by_strategy(scored_df, target_families,
                                k=TOP_K_PER_FAMILY, random_state=RANDOM_STATE):
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

    selections = {'top_n': {}, 'time_diverse': {}, 'time_geo_diverse': {}}

    for family in target_families:
        fam = scored_df[scored_df['family'] == family].copy()

        if fam.empty:
            for s in selections:
                selections[s][family] = []
            continue

        # top_n: rank by raw count
        selections['top_n'][family] = (
            fam.sort_values(['count', 'species'], ascending=[False, True])
               .head(k)['species'].tolist()
        )

        # time_diverse: spread evenly across sorted mean_doy,
        # preferring higher temporal entropy within each DOY band
        fam_td = fam.sort_values(['mean_doy', 't_entropy', 'species'],
                                  ascending=[True, False, True]).reset_index(drop=True)
        m = len(fam_td)
        if m <= k:
            selections['time_diverse'][family] = fam_td['species'].tolist()
        else:
            idx = sorted(set(np.round(np.linspace(0, m - 1, k)).astype(int).tolist()))
            used = set(idx)
            i = 0
            while len(idx) < k and i < m:
                if i not in used:
                    idx.append(i)
                    used.add(i)
                i += 1
            selections['time_diverse'][family] = fam_td.iloc[sorted(idx)]['species'].tolist()

        # time_geo_diverse: cluster on (mean_doy, mean_lat, mean_lon),
        # pick highest composite score representative per cluster
        candidate_pool = min(len(fam), k * 3)
        candidates = (
            fam.sort_values(['count', 'species'], ascending=[False, True])
               .head(candidate_pool).copy().reset_index(drop=True)
        )
        if len(candidates) <= k:
            selections['time_geo_diverse'][family] = candidates['species'].tolist()
        else:
            X = candidates[['mean_doy', 'mean_lat', 'mean_lon']].to_numpy()
            X_scaled = StandardScaler().fit_transform(X)
            n_clusters = min(k, len(candidates))
            km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
            candidates['cluster'] = km.fit_predict(X_scaled)
            selected = (
                candidates.sort_values(['cluster', 'score_time_geo', 'species'],
                                        ascending=[True, False, True])
                           .groupby('cluster', as_index=False).first()
            )
            if len(selected) < k:
                already = set(selected['species'])
                backfill = (
                    candidates[~candidates['species'].isin(already)]
                    .sort_values(['score_time_geo', 'species'], ascending=[False, True])
                    .head(k - len(selected))
                )
                selected = pd.concat([selected, backfill], ignore_index=True)
            selections['time_geo_diverse'][family] = selected['species'].tolist()

    return selections


# ─────────────────────────────────────────────────────────────────────────────
# build_selected_species_df
#   Given one strategy's family→species dict, return the matching
#   slice of the full observation dataframe.
# ─────────────────────────────────────────────────────────────────────────────
def build_selected_species_df(obs_df, strategy_selections):
    all_species = [sp for sp_list in strategy_selections.values() for sp in sp_list]
    return obs_df[obs_df['species'].isin(all_species)].copy().reset_index(drop=True)


print('Helper functions defined.')


## 3. Plotting Helpers

In [ ]:
def plot_species_counts_per_family(eligible_df, title='Eligible species per family'):
    counts = eligible_df.groupby('family')['species'].nunique().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(max(5, len(counts) * 1.2), 4))
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_xlabel('Family'); ax.set_ylabel('# eligible species'); ax.set_title(title)
    plt.tight_layout(); plt.show()


def plot_top_species_counts(eligible_df, target_families, top_n=20):
    n = len(target_families)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]
    for ax, family in zip(axes, target_families):
        fam = (eligible_df[eligible_df['family'] == family]
               .sort_values('count', ascending=False).head(top_n))
        ax.barh(fam['species'][::-1], fam['count'][::-1], color='steelblue')
        ax.set_title(family); ax.set_xlabel('Observation count')
        ax.tick_params(axis='y', labelsize=7)
    fig.suptitle(f'Top {top_n} species by count', fontsize=11, y=1.01)
    plt.tight_layout(); plt.show()


def plot_month_distribution(obs_df, strategy_dict, target_families):
    n = len(target_families)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4))
    if n == 1: axes = [axes]
    for ax, family in zip(axes, target_families):
        sp_list = strategy_dict.get(family, [])
        subset  = obs_df[(obs_df['family'] == family) & (obs_df['species'].isin(sp_list))]
        mc = subset['month'].dropna().value_counts().sort_index()
        ax.bar(mc.index, mc.values, color='darkorange', edgecolor='white')
        ax.set_title(family); ax.set_xlabel('Month'); ax.set_ylabel('Observations')
        ax.set_xticks(range(1, 13))
    fig.suptitle('Month distribution — selected species', fontsize=11, y=1.01)
    plt.tight_layout(); plt.show()


def plot_geo_distribution(obs_df, strategy_dict, target_families):
    n = len(target_families)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4))
    if n == 1: axes = [axes]
    for ax, family in zip(axes, target_families):
        sp_list = strategy_dict.get(family, [])
        subset  = obs_df[
            (obs_df['family'] == family) & (obs_df['species'].isin(sp_list))
        ].dropna(subset=['decimalLatitude', 'decimalLongitude'])
        colors = plt.cm.tab20(np.linspace(0, 1, max(len(sp_list), 1)))
        sp_color = {sp: c for sp, c in zip(sp_list, colors)}
        for sp in sp_list:
            s = subset[subset['species'] == sp]
            ax.scatter(s['decimalLongitude'], s['decimalLatitude'],
                       c=[sp_color[sp]], s=4, alpha=0.4, label=sp)
        ax.set_title(family); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        ax.legend(fontsize=5, markerscale=3, loc='lower right', ncol=2)
    fig.suptitle('Geographic distribution — selected species', fontsize=11, y=1.01)
    plt.tight_layout(); plt.show()


def print_strategy_summary_table(selections, scored_df):
    rows = []
    for strategy, family_dict in selections.items():
        for family, sp_list in family_dict.items():
            if not sp_list:
                rows.append(dict(strategy=strategy, family=family, n=0,
                                  min_count='—', max_count='—'))
                continue
            counts = scored_df.loc[
                (scored_df['family'] == family) & (scored_df['species'].isin(sp_list)), 'count'
            ]
            rows.append(dict(strategy=strategy, family=family, n=len(sp_list),
                              min_count=int(counts.min()), max_count=int(counts.max())))
    tbl = pd.DataFrame(rows)
    print(tbl.to_string(index=False))
    return tbl


print('Plotting helpers defined.')


## 4. Load and Filter Data

In [ ]:
df_raw = load_and_filter_gbif(
    path=DATA_PATH,
    target_families=TARGET_FAMILIES,
    extra_families=OPTIONAL_EXTRA_FAMILIES,
    country=COUNTRY,
    research_grade_only=RESEARCH_GRADE_ONLY,
    require_image=REQUIRE_IMAGE,
)
df_raw.shape


In [ ]:
# Quick sanity check — families present after filtering
df_raw['family'].value_counts()


In [ ]:
# Restrict working df to target families; add time features
df = df_raw[df_raw['family'].isin(TARGET_FAMILIES)].copy()
df = add_time_features(df)

print(f'Working df: {len(df):,} rows across {df["species"].nunique()} species')
df[['family', 'species', 'month', 'doy', 'decimalLatitude', 'decimalLongitude']].head()


## 5. Species Summary Statistics

In [ ]:
species_summary = summarize_species(df)

print(f'Species summary: {len(species_summary)} species')
species_summary.sort_values('count', ascending=False).head(10)


In [ ]:
# Distribution of observation counts — helps choose MIN_OBS_PER_SPECIES
fig, axes = plt.subplots(1, len(TARGET_FAMILIES), figsize=(5 * len(TARGET_FAMILIES), 3))
if len(TARGET_FAMILIES) == 1: axes = [axes]
for ax, fam in zip(axes, TARGET_FAMILIES):
    vals = species_summary[species_summary['family'] == fam]['count']
    ax.hist(vals, bins=40, color='steelblue', edgecolor='white')
    ax.axvline(MIN_OBS_PER_SPECIES, color='red', linestyle='--',
               label=f'threshold={MIN_OBS_PER_SPECIES}')
    ax.set_title(fam); ax.set_xlabel('Observation count'); ax.set_ylabel('# species')
    ax.legend(fontsize=8)
plt.suptitle('Count distribution per species', y=1.01)
plt.tight_layout(); plt.show()
# Adjust MIN_OBS_PER_SPECIES in the config cell if the red line cuts off too many candidates.


## 6. Filter to Eligible Species

In [ ]:
eligible_species_df = filter_eligible_species(
    species_summary,
    min_obs=MIN_OBS_PER_SPECIES,
    min_media_obs=MIN_MEDIA_OBS_PER_SPECIES,
    min_months=MIN_MONTHS,
)

print('\nEligible species per family:')
print(eligible_species_df[eligible_species_df['family'].isin(TARGET_FAMILIES)]
      .groupby('family')['species'].nunique().to_string())


In [ ]:
plot_species_counts_per_family(
    eligible_species_df[eligible_species_df['family'].isin(TARGET_FAMILIES)],
    title='Eligible species per family (after thresholds)',
)


In [ ]:
plot_top_species_counts(
    eligible_species_df[eligible_species_df['family'].isin(TARGET_FAMILIES)],
    TARGET_FAMILIES, top_n=20,
)


## 7. Geo-Only Baseline Accuracy

Mirrors the analysis in `01_eda.ipynb` (cell 23). A quick logistic regression on
lat/lon within each family tests whether geography has discriminative power —
relevant context for how valuable geo priors will be in the later modeling dataset.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

# Restrict to eligible species so the comparison is apples-to-apples
eligible_obs = df[df['species'].isin(eligible_species_df['species'])].copy()

print('Geo-only (lat/lon → species) logistic regression accuracy within each family')
print('(train = test, so these are optimistic — what matters is the ratio vs random baseline)\n')

for family in TARGET_FAMILIES:
    subset = eligible_obs[eligible_obs['family'] == family].dropna(
        subset=['decimalLatitude', 'decimalLongitude', 'species']
    )
    n_species = subset['species'].nunique()
    if n_species < 2:
        print(f'{family}: only {n_species} species — skipping')
        continue

    X = subset[['decimalLatitude', 'decimalLongitude']].values
    y = LabelEncoder().fit_transform(subset['species'])
    score = LogisticRegression(max_iter=500, C=1.0).fit(X, y).score(X, y)
    random_baseline = round(1 / n_species, 4)

    print(
        f'{family}: geo-only accuracy = {score:.2f} | '
        f'random baseline = {random_baseline:.4f} (1/{n_species} species) | '
        f'ratio = {score / random_baseline:.1f}x above random'
    )


## 8. Score and Select Species

In [ ]:
scored_df = score_species(eligible_species_df)

# Sanity check — top species by composite score
scored_df[['family', 'species', 'count', 't_entropy', 'score_count',
           'score_time', 'score_geo', 'score_time_geo']]\
    .sort_values('score_time_geo', ascending=False).head(10)


In [ ]:
selections = select_species_by_strategy(
    scored_df, TARGET_FAMILIES, k=TOP_K_PER_FAMILY, random_state=RANDOM_STATE
)

# Mirror variable names from 01_eda
top_n            = selections['top_n']
time_diverse     = selections['time_diverse']
time_geo_diverse = selections['time_geo_diverse']

print('Selections built. Strategies:', list(selections.keys()))


In [ ]:
# Compact summary: n selected, min/max count per strategy × family
summary_tbl = print_strategy_summary_table(selections, scored_df)

print()
for strategy, family_dict in selections.items():
    print(f'\n── {strategy} ──')
    for family, sp_list in family_dict.items():
        preview = ', '.join(sp_list[:5]) + ('...' if len(sp_list) > 5 else '')
        print(f'  {family} ({len(sp_list)}): {preview}')


## 9. Plots by Strategy

In [ ]:
# Month distributions for time_geo_diverse
plot_month_distribution(df, time_geo_diverse, TARGET_FAMILIES)


In [ ]:
# Geographic scatter for time_geo_diverse
plot_geo_distribution(df, time_geo_diverse, TARGET_FAMILIES)


In [ ]:
# DOY spread comparison across all three strategies (no seaborn)
strategy_colors = {'top_n': 'steelblue', 'time_diverse': 'darkorange', 'time_geo_diverse': 'seagreen'}

fig, axes = plt.subplots(1, len(TARGET_FAMILIES), figsize=(6 * len(TARGET_FAMILIES), 4), sharey=True)
if len(TARGET_FAMILIES) == 1: axes = [axes]

for ax, family in zip(axes, TARGET_FAMILIES):
    for strategy, color in strategy_colors.items():
        sp_list = selections[strategy].get(family, [])
        subset  = df[(df['family'] == family) & (df['species'].isin(sp_list))]
        doy_vals = subset['doy'].dropna()
        if len(doy_vals) == 0: continue
        ax.hist(doy_vals, bins=24, alpha=0.45, color=color, label=strategy, density=True)
    ax.set_title(family); ax.set_xlabel('Day of Year (approx)'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('DOY distribution by strategy', y=1.01)
plt.tight_layout(); plt.show()


## 10. Build Final Species DataFrame

Using `time_geo_diverse` as the default — edit `CHOSEN_STRATEGY` to switch.

In [ ]:
CHOSEN_STRATEGY = 'time_geo_diverse'  # 'top_n' | 'time_diverse' | 'time_geo_diverse'

time_geo_species_df = build_selected_species_df(df, selections[CHOSEN_STRATEGY])

print(
    f'Final df ({CHOSEN_STRATEGY}): {len(time_geo_species_df):,} observations, '
    f'{time_geo_species_df["species"].nunique()} species'
)
time_geo_species_df[['family', 'species', 'month', 'decimalLatitude', 'decimalLongitude']].head()


In [ ]:
# Per-species observation count in the final set
obs_counts = (
    time_geo_species_df
    .groupby(['family', 'species'])['species']
    .count().rename('n_obs')
    .sort_values(ascending=False)
    .reset_index()
)
print(f'Total observations: {obs_counts["n_obs"].sum():,}')
obs_counts


## 11. Export

In [ ]:
# 1. Full species summary
species_summary.to_csv(os.path.join(OUTPUT_DIR, 'species_summary_us.csv'), index=False)
print('Saved species_summary_us.csv')

# 2. Eligible species (post-threshold)
eligible_species_df.to_csv(os.path.join(OUTPUT_DIR, 'eligible_species_us.csv'), index=False)
print('Saved eligible_species_us.csv')

# 3. Selected species by strategy (long format: strategy, family, species)
sel_rows = [
    {'strategy': strategy, 'family': family, 'species': sp}
    for strategy, family_dict in selections.items()
    for family, sp_list in family_dict.items()
    for sp in sp_list
]
pd.DataFrame(sel_rows).to_csv(
    os.path.join(OUTPUT_DIR, 'selected_species_by_strategy.csv'), index=False
)
print('Saved selected_species_by_strategy.csv')

# 4. Final chosen set (full observation rows for the chosen strategy)
final_path = os.path.join(OUTPUT_DIR, f'final_species_{CHOSEN_STRATEGY}.csv')
time_geo_species_df.to_csv(final_path, index=False)
print(f'Saved final_species_{CHOSEN_STRATEGY}.csv  ({len(time_geo_species_df):,} rows)')


## 12. Interpretation Prompts

Work through these questions before committing to a species set for the modeling dataset.

---

**Which families remain viable after thresholds?**  
Check the eligible-species-per-family bar chart. If a family has fewer than 5–10 eligible
species, the `TOP_K_PER_FAMILY` target may not be achievable — lower the threshold or drop that family.

**Which species are abundant but geographically narrow?**  
Look at `lat_range` and `lon_range` in `eligible_species_df`. High `count` + low `lat_range` / `lon_range`
= likely a regional endemic. Good for data volume but poor for geo-prior learning.

**Which species are geographically broad but seasonally narrow?**  
Check `t_entropy` and `n_months`. A species with `n_months == 2` is almost certainly
not useful for seasonal modeling even if it appears across many states.

**Does `time_geo_diverse` seem like the best compromise?**  
Compare the DOY-distribution plot across strategies. If `time_geo_diverse` and `time_diverse`
look nearly identical, the geo component may not be adding much — revisit the composite
score weights in `score_species()`.

**Is the geo-only accuracy meaningfully above baseline?**  
If the ratio is close to 1× for a family, geography alone is not discriminative there.
That doesn't mean geo priors are useless combined with image features, but worth noting.

**Are observation counts balanced enough across families?**  
If one family dominates 5–10×, reduce `TOP_K_PER_FAMILY` for that family or raise its
`MIN_OBS_PER_SPECIES` threshold before finalizing the dataset.